In [1]:
import sqlite3
import pandas as pd
import numpy as np
import pickle
import re
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

print("Libraries imported successfully")

Libraries imported successfully


# Tags Alterations

In [ ]:
conn = sqlite3.connect('../data/tags_alterations_full_2025-10_v2.db')
df = pd.read_sql_query("SELECT * FROM tag_inconsistencies", conn)

print(f"Loaded {len(df)} tag inconsistencies from database")

In [3]:
df = df.drop(columns=['id'])

In [4]:
for col in ['old_snap_timestamp', 'old_rev_timestamp', 'new_snap_timestamp', 'new_rev_timestamp', 'min_delta']:
    df[col] = pd.to_datetime(df[col], unit='s', errors='coerce')

In [5]:
df['snap_year'] = df['new_snap_timestamp'].dt.year
df['rev_year'] = df['new_rev_timestamp'].dt.year
df['category'] = df['new_revision'].apply(lambda x: 'Move' if pd.notna(x) else 'Deletion')

In [6]:
df['delta'] = np.where(
    (df['new_snapshot_cpt'] - df['old_snapshot_cpt']) < 2,
    0,
    (df['min_delta'] - df['old_snap_timestamp']).dt.days
)

In [7]:
def categorize_platform(origin_url):
    if pd.isna(origin_url):
        return 'Other'
    if re.match(r'^https?://[^/]*github\.com', origin_url):
        return 'GitHub'
    elif re.match(r'^https?://[^/]*gitlab\.', origin_url):
        return 'GitLab'
    elif re.match(r'^https?://android\.googlesource\.com', origin_url):
        return 'Android'
    elif re.match(r'^https?://[^/]*bitbucket\.', origin_url):
        return 'BitBucket'
    elif re.match(r'^https?://[^/]*codeberg\.', origin_url):
        return 'Codeberg'
    elif re.match(r'^https?://[^/]*git\.sr\.ht', origin_url):
        return 'SourceHut'
    else:
        return 'Other'

df['platform'] = df['origin_url'].apply(categorize_platform)

In [8]:
invalid_snapshots = df[df['old_snap_timestamp'] >= df['new_snap_timestamp']]

print(f"Total inconsistencies: {len(df)}")
print(f"Cases where old_snap_timestamp > new_snap_timestamp: {len(invalid_snapshots)}")
print(f"Valid snapshot ordering: {len(df) - len(invalid_snapshots)} ({100 * (len(df) - len(invalid_snapshots)) / len(df):.2f}%)")

if len(invalid_snapshots) > 0:
    print("\nSample of invalid cases:")
    print(invalid_snapshots[['origin_url', 'tag_name', 'old_snap_timestamp', 'new_snap_timestamp']].head(10))
else:
    print("\n✓ All old_snapshot timestamps are older than their corresponding new_snapshot timestamps")

Total inconsistencies: 15166153
Cases where old_snap_timestamp > new_snap_timestamp: 1
Valid snapshot ordering: 15166152 (100.00%)

Sample of invalid cases:
                                              origin_url          tag_name  \
11005277  https://github.com/cornerstonejs/cornerstone3D  refs/tags/v2.2.3   

          old_snap_timestamp  new_snap_timestamp  
11005277 2024-11-12 18:40:57 2024-11-12 18:40:57  


In [9]:
print(f"length of df : {len(df):_}")
df.head(5)

length of df : 15_166_153


,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform
0,https://github.com/elevenlee/oop_generics,refs/tags/v1.0,lightweight,swh:1:snp:589750d264ea6ba72cd290d154b089ce79c20b14,0,2015-08-05 00:56:29,swh:1:rev:16f2c36c23dc1e432a183f2575be1e2b4d654ca9,2012-12-29 07:00:19,swh:1:dir:e482b7c2c58ab76031e45dd163c159e1a46b70a2,swh:1:snp:e613bc07696f35d8c1d341bd9928a313d3a705ac,1,2016-03-14 02:41:49,NaN,NaT,NaN,2015-08-05 00:56:29,2016,NaN,Deletion,0,GitHub
1,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.0.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:29e7fce9fc88a55bdafb507f6506bd6f153260b0,2020-04-27 03:35:14,swh:1:dir:c612010c922cea11ecdfa384db474123a0225d73,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub
2,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.2.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:16a7fefff1f1fb674d1ed0328099a8d6ac0d38bb,2020-04-27 19:31:38,swh:1:dir:3727b13bc24dad71024adebc98eb17a2b9310d35,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub
3,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/3.0.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:85401c9f014ff7d8950b1930e732b73fd6d08043,2020-05-07 07:24:36,swh:1:dir:65029e9180c14f7e3c5d0d1d3cf0b9b697b5e60e,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub
4,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.3.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:6fa41ccd9ece1292f0df3554e0c5f6680c82e818,2020-04-29 08:42:15,swh:1:dir:1edad0e47774c0fb05657616f6d8a4210cbcacb0,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub


## Stars

In [10]:
stars = pd.read_pickle("/home/infres/rapaport/datasets/github-stars/github-stars.pkl")

In [11]:
stars['origin'] = stars['origin'].str.strip('"')
stars = stars.drop_duplicates(subset='origin')

In [12]:
df_bis = df.copy()

In [13]:
df_bis = df_bis.merge(stars[['origin', 'stars']], 
              left_on='origin_url', 
              right_on='origin', 
              how='left')

print(f"DataFrame shape after merge: {df_bis.shape}")
print(f"Number of rows with stars data: {df_bis['stars'].notna().sum():_}")
print(f"Number of rows without stars data (null): {df_bis['stars'].isna().sum():_}")

DataFrame shape after merge: (15166153, 23)
Number of rows with stars data: 12_381_594
Number of rows without stars data (null): 2_784_559


In [14]:
df_bis = df_bis.drop(columns=['origin'])

In [15]:
print(f"Initial: {len(df_bis):_}")
df_dd = df_bis.drop_duplicates().reset_index(drop=True)
print(f"Mid: {len(df_dd):_}")
df_min = df_bis[['origin_url', 'tag_name', 'type', 'old_snapshot', 'old_snap_timestamp']]
print(f"Init min {len(df_min):_}")
df_min_dd = df_min.drop_duplicates().reset_index(drop=True)
print(f"end: {len(df_min_dd):_}")

Initial: 15_166_153
Mid: 15_166_153
Init min 15_166_153
end: 15_166_153


In [16]:
df[(df['origin_url']=="https://git.codelinaro.org/clo/la/platform/external/sepolicy.git") & (df['tag_name']=='refs/tags/android-4.4.1_r1') & (df['old_snapshot'] == "swh:1:snp:fa0ade8bea3fb0a1421f99889b42a0e913dedda7")]

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform
15160952,https://git.codelinaro.org/clo/la/platform/external/sepolicy.git,refs/tags/android-4.4.1_r1,annotated,swh:1:snp:fa0ade8bea3fb0a1421f99889b42a0e913dedda7,0,2023-08-23 13:01:47,swh:1:rev:0ad5707da7fc6052b0ec170a33dde6f924a6ab81,2013-11-15 20:01:37,swh:1:dir:00bd31e3cf24a72d3250a36f8eed6e6e48ee161e,swh:1:snp:299cc0812d4bce77b458f1c0161af6d5964d2e19,3,2024-02-20 04:30:38,swh:1:rev:35e8dcc9ba40c6419f63d0a516c0995d3064f96e,2013-11-15 00:19:25,swh:1:dir:acdfd6e8da1a74a21c83d766708033c2eb11be1d,2023-11-23 12:20:49,2024,2013.0,Alteration,91,Other
15160954,https://git.codelinaro.org/clo/la/platform/external/sepolicy.git,refs/tags/android-4.4.1_r1,annotated,swh:1:snp:fa0ade8bea3fb0a1421f99889b42a0e913dedda7,4,2024-05-20 06:03:24,swh:1:rev:0ad5707da7fc6052b0ec170a33dde6f924a6ab81,2013-11-15 20:01:37,swh:1:dir:00bd31e3cf24a72d3250a36f8eed6e6e48ee161e,swh:1:snp:299cc0812d4bce77b458f1c0161af6d5964d2e19,8,2025-05-10 17:20:33,swh:1:rev:35e8dcc9ba40c6419f63d0a516c0995d3064f96e,2013-11-15 00:19:25,swh:1:dir:acdfd6e8da1a74a21c83d766708033c2eb11be1d,2025-02-10 14:30:21,2025,2013.0,Alteration,266,Other


In [17]:
with open("../data/tag_alterations_full_2025-10_v3.pkl", "wb") as f:
    pickle.dump(df_bis, f)

In [16]:
df_bis.to_sql("tags_with_stars", conn, if_exists='replace', index=False)

15166153

In [17]:
df_bis.sample(2)

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars
11757355,https://github.com/f8f8f8ff/qmk_firmware,refs/tags/0.10.33,lightweight,swh:1:snp:c7d4da4761386578f087cc0ef4f259f5dfc38760,0,2023-10-23 05:35:04,swh:1:rev:459f672879e99f27faadb548b6493760b9a5879c,2020-10-09 17:46:49,swh:1:dir:f5a72eddb3608059b8f89c1519aefd91b9fef376,swh:1:snp:66c5f0b6667b2ccf9d169c74abc67561d62e443d,1,2024-10-11 11:44:33,NaN,NaT,NaN,2023-10-23 05:35:04,2024,NaN,Deletion,0,GitHub,0.0
944265,https://github.com/stephensmalley/selinux-kernel,refs/tags/v4.4.119,annotated,swh:1:snp:4c7856eeed86dd193c8b9312f8da38236eabfd17,10,2024-07-12 18:42:21,swh:1:rev:5e0c4113fcba3b47e9b827f9f661d85cc238b5a6,2018-02-28 09:17:24,swh:1:dir:344a62a2685509c01779e6516e0757c0ca992ed1,swh:1:snp:5122fa6512264da76f44465a213337de7cc63f7e,31,2025-09-12 20:54:04,NaN,NaT,NaN,2025-08-29 07:08:45,2025,NaN,Deletion,412,GitHub,9.0


# Deletion --> Move

In [18]:
conn = sqlite3.connect('../data/tags_alterations_full_2025-10_v2.db')
dm = pd.read_sql_query("SELECT * FROM deletion_creation_v2", conn)

print(f"Loaded {len(dm):_} tag inconsistencies from database")

Loaded 4_865_355 tag inconsistencies from database


In [19]:
for col in ['old_snap_timestamp', 'new_snap_timestamp', 'creation_snap_ts', 'creation_rev_ts', 'creation_delta']:
    dm[col] = pd.to_datetime(dm[col], unit='s', errors='coerce')

In [20]:
dm.head(2)

,origin_url,tag_name,type,old_snapshot,old_snap_timestamp,old_revision,old_root_dir,new_snapshot,new_snap_timestamp,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta
0,https://github.com/elevenlee/oop_generics,refs/tags/v1.0,lightweight,swh:1:snp:589750d264ea6ba72cd290d154b089ce79c20b14,2015-08-05 00:56:29,swh:1:rev:16f2c36c23dc1e432a183f2575be1e2b4d654ca9,swh:1:dir:e482b7c2c58ab76031e45dd163c159e1a46b70a2,swh:1:snp:e613bc07696f35d8c1d341bd9928a313d3a705ac,2016-03-14 02:41:49,annotated,swh:1:rev:16f2c36c23dc1e432a183f2575be1e2b4d654ca9,2012-12-29 07:00:19,swh:1:dir:e482b7c2c58ab76031e45dd163c159e1a46b70a2,e613bc07696f35d8c1d341bd9928a313d3a705ac,2016-03-14 02:41:49,2016-03-14 02:41:49
1,https://github.com/edgarwang/maze-game,refs/tags/v1.0.0,lightweight,swh:1:snp:c55b831ebb9e54ae295336bd3cc804646de830b4,2015-08-09 21:48:47,swh:1:rev:72256857b6beab53b85bf29deb0bf10bb406de34,swh:1:dir:219a52d9e416780304d3d5c0fc7aaa0ee0a04a4f,swh:1:snp:97431b312d0481e95e78ba01598fc742f82db33f,2016-03-14 19:52:14,annotated,swh:1:rev:72256857b6beab53b85bf29deb0bf10bb406de34,2013-12-29 04:33:40,swh:1:dir:219a52d9e416780304d3d5c0fc7aaa0ee0a04a4f,97431b312d0481e95e78ba01598fc742f82db33f,2016-03-14 19:52:14,2016-03-14 19:52:14


In [21]:
dm[dm['creation_delta']<dm['creation_snap_ts']]

,origin_url,tag_name,type,old_snapshot,old_snap_timestamp,old_revision,old_root_dir,new_snapshot,new_snap_timestamp,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta
729,https://github.com/vasiliy-t/tarantool-operator,refs/tags/tarantool-operator-0.0.8,lightweight,swh:1:snp:c5c5ae6ba9e33a3f660a08b021c9dae1c7b2d88a,2020-09-13 10:38:35,swh:1:rev:0ece70d8cfc8c68310012254d1eb95491c1df925,swh:1:dir:f487c64e71d8bc4a0d7d7600f27d23ff08b7a631,swh:1:snp:ead4b2e34963bd7ed9a7eb1b111f183cd1dabb54,2021-07-10 06:29:30,annotated,swh:1:rev:1d9a49a9f8f46b66e1479b418982ec0a2612386c,2020-12-16 12:14:45,swh:1:dir:311fec4eb066fe28f0712769ee49f9114b7ae39b,4062b2d1e72f9b5bffc750fbe8ab9e91dcf4aa58,2022-04-04 10:49:57,2021-07-10 06:29:30
5516,https://github.com/eclipse-zenoh/zenoh,refs/tags/1.0.0-alpha.4,annotated,swh:1:snp:8eddbffc600ba1bb2aae717859a4d785ed8f0a8b,2024-07-12 21:05:57,swh:1:rev:3f42328696bac0abced16c179f38374b7453068f,swh:1:dir:77415747c8b7df86bfb5122bc4e3e0bb6dbc6a65,swh:1:snp:48ca1b510022ce33b8a0efb346cc809e81225f5f,2024-08-05 06:20:02,lightweight,swh:1:rev:3f42328696bac0abced16c179f38374b7453068f,2024-07-10 10:06:56,swh:1:dir:77415747c8b7df86bfb5122bc4e3e0bb6dbc6a65,cab57ee0fab740dbce0b9922b7b1d3c72d7ee3b2,2024-08-30 13:10:02,2024-08-05 06:20:02
8988,https://github.com/BrightspaceHypermediaComponents/sequences,refs/tags/v3.0.0,lightweight,swh:1:snp:0559cdf4c31dcfadf73a55665392883f377b54e3,2021-06-06 20:14:47,swh:1:rev:85091531da150575ba6c3f7a7faa9e7cffb8dae0,swh:1:dir:d395da40ee0de36a1b53dae02fe6d4ff279d45e7,swh:1:snp:f07f9d425f3197661d1ecbbbcdbdc16a2c4c9b18,2022-01-31 20:32:21,annotated,swh:1:rev:b66f56828b8c9224496b93ae33b24371a7ec1610,2022-03-22 14:21:45,swh:1:dir:eccc1aceb5a4ff588b39d9b8b8aa929ac5f821cc,7b72153d6cde23e993d32af0484a0319c5e67acf,2022-04-28 03:42:11,1970-02-04 11:59:15
11089,https://github.com/joinbox/jb-backoffice-forms,refs/tags/v0.3.6,lightweight,swh:1:snp:1eed265a7732a208d16fc77e6227378586c81295,2015-08-21 00:58:04,swh:1:rev:01cfc1cc6e10d6f2f907fdb0fe553b1f658b3a6f,swh:1:dir:f4777bd977a2f5fe4277de5a82e8f0aa18cb5022,swh:1:snp:bbbc64a74f29db71b91d915a05153fa583f35c15,2016-03-29 21:47:09,annotated,swh:1:rev:b26f93206ad0801c67ab8910080b35ffb6e6791d,2016-11-24 17:25:54,swh:1:dir:acc5300761dba62eafc17e62009445762fe0e9e9,f9f1427734fa11a2bc71d883d57cce62f06845fa,2017-09-24 07:34:07,1970-06-14 04:50:25
33007,https://github.com/phattrien/Nukeviet,refs/tags/4.0.15,lightweight,swh:1:snp:7579003946a51f22713e616dcc157fc34f15fb8b,2015-07-25 18:22:46,swh:1:rev:477c6e10e617ed953dad7f0fa3a946e76b7f61fb,swh:1:dir:8228c22b7abce64af159489508c8f87509135c74,swh:1:snp:e9e4230efdc2075368ed333952cbca7bd32c5fc6,2016-02-27 14:17:32,annotated,swh:1:rev:477c6e10e617ed953dad7f0fa3a946e76b7f61fb,2015-04-23 16:01:42,swh:1:dir:8228c22b7abce64af159489508c8f87509135c74,5aeb783d9017dbf61e096ad02b1af3108b5ce058,2016-05-24 04:15:50,1970-02-22 17:34:16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4852908,https://github.com/skywalker512/core,refs/tags/v0.1.0-beta.7.1,lightweight,swh:1:snp:42634271325d340c78f245413118819991f2f4ef,2018-03-26 01:29:42,swh:1:rev:1dc0a44096093bd14d689eccffdf39c75b8c303a,swh:1:dir:823a040e9aa9b2166ce653c6b2108edf39f7ea9f,swh:1:snp:a66b5ca1e3432af4ca1750dfd901c4b94baff5fb,2019-02-01 23:42:10,annotated,swh:1:rev:c6aeeeb3c1d5cf92c586c9f8ee43d2127d0fb933,2018-01-06 09:34:42,swh:1:dir:13082a80455b67c4fc62c5628acf20057a26410a,8c3b752e4b6a81294930b3e7477949d03e319732,2019-04-07 18:35:16,2019-02-01 23:42:10
4856059,https://github.com/oxygen-cms/mod-media,refs/tags/0.3.0,annotated,swh:1:snp:b6df4f93b4ba5a55eff584cebb7dad0ce7777611,2016-03-27 04:17:33,swh:1:rev:b04f7621e3c4a3270371ec9b74a69370898477da,swh:1:dir:345081d74d281c5da886e14e095045d41aa88f13,swh:1:snp:cb4dbaa2c9a8b7734c896236d87dee46d5b14d73,2016-04-02 22:14:16,lightweight,swh:1:rev:717e19603986188656635670e33df3e03e4211e5,2020-04-07 06:13:43,swh:1:dir:a4dc0d1c668bf137b09062a460e20b174854bcb1

In [21]:
dm['deadtime'] = np.where(
    dm['creation_delta'] == dm['creation_snap_ts'],
    pd.Timedelta(0),
    np.where(
        dm['creation_delta'] < dm['creation_snap_ts'],
        dm['creation_delta'] - dm['new_snap_timestamp'],
        pd.NaT
    )
)

print(f"Created 'deadtime' column")
print(f"Deadtime = 0: {(dm['deadtime'] == pd.Timedelta(0)).sum():_}")
print(f"Deadtime calculated: {(dm['deadtime'].notna() & (dm['deadtime'] != pd.Timedelta(0))).sum():_}")
print(f"Deadtime = NA: {dm['deadtime'].isna().sum():_}")

Created 'deadtime' column
Deadtime = 0: 4_862_417
Deadtime calculated: 2_938
Deadtime = NA: 0


In [22]:
dm['status'] = np.where(
    (dm['creation_delta'] == dm['new_snap_timestamp']) & 
    (dm['type'] == "lightweight") & 
    (dm['creation_type'] == "annotated") &
    (dm['old_snap_timestamp'] < pd.Timestamp('2015-09-18')),
    'non-legit',
    'legit'
)

print(f"Status column created:")
print(f"Non-legit: {(dm['status'] == 'non-legit').sum():_}")
print(f"Legit: {(dm['status'] == 'legit').sum():_}")

Status column created:
Non-legit: 4_348_849
Legit: 516_506


In [23]:
merge_keys = ['origin_url', 'tag_name', 'type', 'old_snapshot', 'old_snap_timestamp', 'old_revision', 'old_root_dir', 'new_snapshot', 'new_snap_timestamp']

dm_check = dm[merge_keys].drop_duplicates()
df_bis_check = df_bis[merge_keys].drop_duplicates()

print(f"Unique combinations in dm: {len(dm_check):_}")
print(f"Unique combinations in df_bis: {len(df_bis_check):_}")

dm_in_df_bis = dm_check.merge(df_bis_check, on=merge_keys, how='inner')
print(f"dm rows that match df_bis: {len(dm_in_df_bis):_} ({100*len(dm_in_df_bis)/len(dm_check):.2f}%)")

Unique combinations in dm: 4_865_355
Unique combinations in df_bis: 15_166_153
dm rows that match df_bis: 4_865_355 (100.00%)


In [24]:
df_bis_merged = df_bis.merge(
    dm.drop(columns=['origin', 'stars'] if 'origin' in dm.columns and 'stars' in dm.columns else []),
    on=merge_keys,
    how='left',
    suffixes=('', '_dm')
)

print(f"\nOriginal df_bis shape: {df_bis.shape}")
print(f"Merged df_bis shape: {df_bis_merged.shape}")
print(f"Rows with dm data: {df_bis_merged['status'].notna().sum():_}")


Original df_bis shape: (15166153, 22)
Merged df_bis shape: (15166153, 31)
Rows with dm data: 4_865_355


In [26]:
with open("../data/tags_alteration_full_2025-10_dm.pkl", "wb") as f:
    pickle.dump(df_bis_merged, f)

In [26]:
df_bis_merged.to_sql("tags_with_deletion_creation_detection", conn, if_exists='replace', index=False)

/tmp/ipykernel_1109004/4149583380.py:1: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  df_bis_merged.to_sql("tags_with_deletion_creation_detection", conn, if_exists='replace', index=False)


15166153

In [25]:
df_bis_merged[df_bis_merged['status'] == "legit"].sample(3)

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta,deadtime,status
9422973,https://github.com/neos/setup,refs/tags/1.0.0,lightweight,swh:1:snp:ed47f0bfd0e8b5323f879bf00d4e70ccfc5a97bd,0,2015-09-25 18:17:44,swh:1:rev:d9b729c49a4b4681959338773ba7c629e20bfb3a,2013-12-10 16:42:21,swh:1:dir:854acb40aa893cf7094d8005f2c9e36c5b6049bc,swh:1:snp:798bae82d27f012ba283a5b44909648f187de7d1,1,2016-03-15 23:41:47,NaN,NaT,NaN,2015-09-25 18:17:44,2016,NaN,Deletion,0,GitHub,6.0,annotated,swh:1:rev:d9b729c49a4b4681959338773ba7c629e20bfb3a,2013-12-10 16:42:21,swh:1:dir:854acb40aa893cf7094d8005f2c9e36c5b6049bc,798bae82d27f012ba283a5b44909648f187de7d1,2016-03-15 23:41:47,2016-03-15 23:41:47,0 days,legit
10520478,https://github.com/seanhoughton/kstars,refs/tags/v4.3.85,lightweight,swh:1:snp:8c4ae406704cdeeed23573bb291a39b78ffd0eff,0,2015-09-25 14:30:34,swh:1:rev:a444425e3da64ae9cd96e212ddcadde1c75e4c97,2009-12-17 01:19:04,swh:1:dir:90d8b605c655f0767ad0fcfdaa5005cf7bd67e74,swh:1:snp:8febe34bf7b503ccc67c3a77ca86bda4870f58ff,1,2016-03-07 18:25:47,NaN,NaT,NaN,2015-09-25 14:30:34,2016,NaN,Deletion,0,GitHub,0.0,annotated,swh:1:rev:a444425e3da64ae9cd96e212ddcadde1c75e4c97,2009-12-17 01:19:04,swh:1:dir:90d8b605c655f0767ad0fcfdaa5005cf7bd67e74,8febe34bf7b503ccc67c3a77ca86bda4870f58ff,2016-03-07 18:25:47,2016-03-07 18:25:47,0 days,legit
10603930,https://github.com/pichina/lk,refs/tags/AU_LINUX_ANDROID_LNX.LA.3.5.1.04.04.02.032.075,lightweight,swh:1:snp:8842d4b0644dcbcd891a8fbd1090706d7beb481a,0,2015-09-25 08:36:52,swh:1:rev:a52897b56e3cfb4a60028c5e0f804eb1f58d2a11,2014-04-04 13:09:23,swh:1:dir:aec5270d8c010488f473b0caddf2349bdf302f9f,swh:1:snp:8b16b962c6b7f231901a3242c5f9a96f004e5970,1,2016-03-14 15:11:18,NaN,NaT,NaN,2015-09-25 08:36:52,2016,NaN,Deletion,0,GitHub,0.0,annotated,swh:1:rev:a52897b56e3cfb4a60028c5e0f804eb1f58d2a11,2014-04-04 13:09:23,swh:1:dir:aec5270d8c010488f473b0caddf2349bdf302f9f,8b16b962c6b7f231901a3242c5f9a96f004e5970,2016-03-14 15:11:18,2016-03-14 15:11:18,0 days,legit


In [28]:
print(f"initial: {len(dm):_}")
dd = dm.drop_duplicates().reset_index(drop=True)
print(f"End: {len(dd):_}")

initial: 4_865_355
End: 4_865_355


In [29]:
with open("../data/deletion_move.pkl", "wb") as f:
    pickle.dump(dm, f)

In [31]:
df_bis_merged[df_bis_merged['category']=="Alteration"]

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta,deadtime,status
150,https://github.com/danielbarela/mage-server,refs/tags/5.2.1,lightweight,swh:1:snp:66cd360e06815f5e39e9281d29df5ec96e2ebeb2,0,2018-10-27 02:37:45,swh:1:rev:575ecf06665e97fb0550d9fed5edfe97e3816f36,2018-03-13 14:46:58,swh:1:dir:7b7103aa062f1461223079cce09f7ede0e9a80e9,swh:1:snp:54218774871910936eb3666aa4cc87947a8d4b23,2,2021-02-13 20:26:14,swh:1:rev:47f388fed1e1042f78bdd05b201d80d328616af2,2018-10-25 17:25:46,swh:1:dir:815dd38f9e6613600b040222ab6e0b99d334cbbd,2019-04-29 04:08:56,2021,2018.0,Alteration,184,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN
152,https://github.com/danielbarela/mage-server,refs/tags/5.2.3,lightweight,swh:1:snp:66cd360e06815f5e39e9281d29df5ec96e2ebeb2,0,2018-10-27 02:37:45,swh:1:rev:124a678bd41d5d910cb4cf11edfded2eb8614d92,2018-03-13 15:04:11,swh:1:dir:add4087a18a8ccaec7374788c6a8cb188c8fbad7,swh:1:snp:54218774871910936eb3666aa4cc87947a8d4b23,2,2021-02-13 20:26:14,swh:1:rev:a01814ef6efb59c115df2c9c5f648b59b37c02b3,2018-12-07 18:39:34,swh:1:dir:2adfa5516ca35afe2c9e8dbfc32c943b2892be42,2019-04-29 04:08:56,2021,2018.0,Alteration,184,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN
157,https://github.com/danielbarela/mage-server,refs/tags/5.2.5,lightweight,swh:1:snp:66cd360e06815f5e39e9281d29df5ec96e2ebeb2,0,2018-10-27 02:37:45,swh:1:rev:31c5df3739f0aff8c72ea548f8fd0f0f807523af,2018-03-13 15:38:10,swh:1:dir:fbe630f2ea0b2111a29f6e61faf92f1081bf35ad,swh:1:snp:54218774871910936eb3666aa4cc87947a8d4b23,2,2021-02-13 20:26:14,swh:1:rev:674bb7686d4f812255a0bb7c12278641ba38f708,2019-03-13 18:21:49,swh:1:dir:69283ab373864173d2b97e8fc16ce409befd257a,2019-04-29 04:08:56,2021,2019.0,Alteration,184,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN
158,https://github.com/danielbarela/mage-server,refs/tags/5.2.4,lightweight,swh:1:snp:66cd360e06815f5e39e9281d29df5ec96e2ebeb2,0,2018-10-27 02:37:45,swh:1:rev:b29cd5247880a46745782b063c86320d18bc931e,2018-03-13 15:34:05,swh:1:dir:26a5778b8481d397f6c818dc071ec942f8a1e5a8,swh:1:snp:54218774871910936eb3666aa4cc87947a8d4b23,2,2021-02-13 20:26:14,swh:1:rev:78ffbeabac6a4f9c651093af990a8f78e1c3a479,2019-01-14 15:51:22,swh:1:dir:29c5968d5d070a64f9f098f77de82de6045ad3fa,2019-04-29 04:08:56,2021,2019.0,Alteration,184,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN
162,https://github.com/danielbarela/mage-server,refs/tags/5.2.6,lightweight,swh:1:snp:66cd360e06815f5e39e9281d29df5ec96e2ebeb2,0,2018-10-27 02:37:45,swh:1:rev:b8952eda354aecdd2455fa69f12ff45b84222b0f,2018-03-13 15:48:51,swh:1:dir:49ae60a8a73ed906a0e1055fb95a4ba37190572a,swh:1:snp:54218774871910936eb3666aa4cc87947a8d4b23,2,2021-02-13 20:26:14,swh:1:rev:fdf1530c6cbbbf17334d87f4c136a6f850f046fd,2019-04-11 21:19:38,swh:1:dir:3fc6e3918bdc081c2d236772842bd47116a97dcb,2019-04-29 04:08:56,2021,2019.0,Alteration,184,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15166144,https://github.com/SolarFramework/Sample-Slam,refs/tags/SolARSample_SLAM/0.11.0/linux,lightweight,swh:1:snp:ac77d4489204ab4c372931df07dddcbd57f3fac1,11,2022-03-12 18:47:52,swh:1:rev:e7f18b56da103d9631b9334c5ef7adbf3b5acd93,2022-02-15 10:12:14,swh:1:dir:25bc03c7f2070c7cbb51c187cc8caa5d5fea9c26,swh:1:snp:39b322880f9a486195f6e8db088c801b7c437045,13,2022-06-28 15:58:54,swh:1:rev:9f2f33d68545bd375c207a3b4b8b20a057daf765,2022-06-12 16:41:31,swh:1:dir:7abde99a459e0fa77de62477587a4450f7116eea,2022-05-29 07:34:53,2022,2022.0,Alteration,77,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN
15166145,https://github.com/SolarFramework/Sample-Slam,refs/tags/SolARPipeline_SLA

In [35]:
df_bis_merged[(df_bis_merged['status']=="legit")]

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta,deadtime,status
733,https://github.com/membphis/apisix,refs/tags/v0.8,annotated,swh:1:snp:23350c33758fcbee37ee8a67f2cf91bf1a199460,0,2020-06-04 08:13:01,swh:1:rev:9bb2174e0434ea46646bc632b122799f49dcf097,2019-09-29 01:00:23,swh:1:dir:f93a4d8afbe8b32200e53355526a9ffbf06a93e2,swh:1:snp:d27a552c700fb1f901ac5c88e7d140720e8c849d,2,2022-04-11 01:20:54,NaN,NaT,NaN,2021-06-20 02:06:24,2022,NaN,Deletion,380,GitHub,0.0,lightweight,swh:1:rev:b65bb153fa9014f56518c44ef9444084502976a6,2019-09-29 07:55:10,swh:1:dir:ce2c8e405ab26c5c2295dd57cdbcaa08d164ae54,d27a552c700fb1f901ac5c88e7d140720e8c849d,2022-04-11 01:20:54,2022-04-11 01:20:54,0 days,legit
842,https://github.com/projectstorm/serve,refs/tags/v1.4.1,lightweight,swh:1:snp:cb6ae5ef63cfd524fae6ce29cf1028781da77922,0,2015-09-25 05:58:29,swh:1:rev:6312fb693a1c90ce2d642c9cb9c2dde6adeb8f23,2015-07-14 08:59:16,swh:1:dir:c0da319ef4f0d76088e1c13940fb2a0eca930caf,swh:1:snp:c7787e60074b94b548d6481f2637de00b87d8f37,1,2016-03-12 17:36:16,NaN,NaT,NaN,2015-09-25 05:58:29,2016,NaN,Deletion,0,GitHub,NaN,annotated,swh:1:rev:6312fb693a1c90ce2d642c9cb9c2dde6adeb8f23,2015-07-14 08:59:16,swh:1:dir:c0da319ef4f0d76088e1c13940fb2a0eca930caf,c7787e60074b94b548d6481f2637de00b87d8f37,2016-03-12 17:36:16,2016-03-12 17:36:16,0 days,legit
843,https://github.com/projectstorm/serve,refs/tags/v1.1.0,lightweight,swh:1:snp:cb6ae5ef63cfd524fae6ce29cf1028781da77922,0,2015-09-25 05:58:29,swh:1:rev:3d3cab6b491ec99ba6772938213183341e16ee5e,2015-05-28 17:47:53,swh:1:dir:9fc6ee00b07998fa73b999126d90964df667fd05,swh:1:snp:c7787e60074b94b548d6481f2637de00b87d8f37,1,2016-03-12 17:36:16,NaN,NaT,NaN,2015-09-25 05:58:29,2016,NaN,Deletion,0,GitHub,NaN,annotated,swh:1:rev:3d3cab6b491ec99ba6772938213183341e16ee5e,2015-05-28 17:47:53,swh:1:dir:9fc6ee00b07998fa73b999126d90964df667fd05,c7787e60074b94b548d6481f2637de00b87d8f37,2016-03-12 17:36:16,2016-03-12 17:36:16,0 days,legit
844,https://github.com/projectstorm/serve,refs/tags/v1.4.2,lightweight,swh:1:snp:cb6ae5ef63cfd524fae6ce29cf1028781da77922,0,2015-09-25 05:58:29,swh:1:rev:4b42db226c9dd48a7332edb85b2b148968c2fd48,2015-07-25 14:42:53,swh:1:dir:e3e6580addd39667a1af073b901004adad69e6c4,swh:1:snp:c7787e60074b94b548d6481f2637de00b87d8f37,1,2016-03-12 17:36:16,NaN,NaT,NaN,2015-09-25 05:58:29,2016,NaN,Deletion,0,GitHub,NaN,annotated,swh:1:rev:4b42db226c9dd48a7332edb85b2b148968c2fd48,2015-07-25 14:42:53,swh:1:dir:e3e6580addd39667a1af073b901004adad69e6c4,c7787e60074b94b548d6481f2637de00b87d8f37,2016-03-12 17:36:16,2016-03-12 17:36:16,0 days,legit
1108,https://github.com/InTheNow/scala-bricks,refs/tags/v0.0.0,lightweight,swh:1:snp:1eadff41c8af217eff0142aa49c0c2f4eaa84946,0,2015-09-26 03:00:14,swh:1:rev:6ef9e7640cd380c081ced28697b5aff85bd11a5c,2015-09-05 16:33:29,swh:1:dir:8d3c5de70419db14a30974ee26e23401bf3b09bf,swh:1:snp:779c948f19d8ff748d946ac98e64c6604595e6ec,1,2016-03-04 03:29:55,NaN,NaT,NaN,2015-09-26 03:00:14,2016,NaN,Deletion,0,GitHub,NaN,annotated,swh:1:rev:6ef9e7640cd380c081ced28697b5aff85bd11a5c,2015-09-05 16:33:29,swh:1:dir:8d3c5de70419db14a30974ee26e23401bf3b09bf,779c948f19d8ff748d946ac98e64c6604595e6ec,2016-03-04 03:29:55,2016-03-04 03:29:55,0 days,legit
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15165675,https://github.com/wavefrontHQ/ruby-client,refs/tags/v1.0.2,lightweight,swh:1:snp:0a55114163489b669aadcb0878430ceb69a42a98,0,2015-09-24 22:42:14,swh:1:rev:04741e540cc430ecb149e9f3877b897e8595bd37,2015-08-28 21:58:43,swh:1:dir:93451ff3d24dc5080f38022b522269a7121b3de2,swh:1:snp:529429b58be5476d9d54be5b096a992b9476a76a,1,2016

In [2]:
# Debug: Check the table contents
conn = sqlite3.connect('../data/tags_alterations_full_2025-10_v2.db')

# Check if table exists
table_check = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='tags_with_deletion_creation_detection'",
    conn
)
print(f"Table exists: {len(table_check) > 0}")

if len(table_check) > 0:
    # Total count
    total = pd.read_sql_query("SELECT COUNT(*) as cnt FROM tags_with_deletion_creation_detection", conn)
    print(f"\nTotal rows in table: {total['cnt'][0]:_}")
    
    # Check status distribution
    status_dist = pd.read_sql_query("SELECT status, COUNT(*) as cnt FROM tags_with_deletion_creation_detection GROUP BY status", conn)
    print(f"\nStatus distribution:")
    print(status_dist)
    
    # Check category distribution
    category_dist = pd.read_sql_query("SELECT category, COUNT(*) as cnt FROM tags_with_deletion_creation_detection GROUP BY category", conn)
    print(f"\nCategory distribution:")
    print(category_dist)
    
    # Check how many have non-null root dirs
    root_dir_check = pd.read_sql_query("""
        SELECT 
            COUNT(*) as total,
            SUM(CASE WHEN new_root_dir IS NOT NULL THEN 1 ELSE 0 END) as new_root_dir_not_null,
            SUM(CASE WHEN creation_root_dir IS NOT NULL THEN 1 ELSE 0 END) as creation_root_dir_not_null,
            SUM(CASE WHEN new_root_dir IS NOT NULL OR creation_root_dir IS NOT NULL THEN 1 ELSE 0 END) as either_not_null
        FROM tags_with_deletion_creation_detection
    """, conn)
    print(f"\nRoot directory statistics:")
    print(root_dir_check)
    
    # Check the actual query condition
    filtered = pd.read_sql_query("""
        SELECT COUNT(*) as cnt 
        FROM tags_with_deletion_creation_detection
        WHERE (status != 'non-legit' OR status IS NULL) 
        AND (creation_root_dir IS NOT NULL OR new_root_dir IS NOT NULL)
    """, conn)
    print(f"\nRows matching your filter: {filtered['cnt'][0]:_}")
    
    # Show sample of what we're trying to get
    sample = pd.read_sql_query("""
        SELECT origin_url, tag_name, type, category, status, 
               CASE WHEN new_root_dir IS NOT NULL THEN 'YES' ELSE 'NO' END as has_new_root_dir,
               CASE WHEN creation_root_dir IS NOT NULL THEN 'YES' ELSE 'NO' END as has_creation_root_dir
        FROM tags_with_deletion_creation_detection
        LIMIT 5
    """, conn)
    print(f"\nSample of 5 rows:")
    print(sample)

conn.close()

Table exists: True

Total rows in table: 15_166_153


KeyboardInterrupt: 

In [3]:
conn.close()